# Dataset Prep

The following cells clean and prepare the CIPHAR dataset.

In [1]:
####################################################################
# Import the required libraries and set our global variables.
####################################################################
import os
import zipfile
import py7zr
import torch
import torchvision.transforms.v2 as transforms
from PIL import Image

# Dataset paths
dataset_archive = 'cifar-10.zip'
dataset_path = 'cifar-10/'

train_dataset_archive = dataset_path + 'train.7z'
train_dataset_path = dataset_path + 'train/'

In [2]:
####################################################################
# Extract the dataset if it has not already been extracted.
####################################################################
if not os.path.exists(dataset_path):
	with zipfile.ZipFile(dataset_archive, 'r') as archive:
		archive.extractall(dataset_path)
		
if not os.path.exists(train_dataset_path):
	with py7zr.SevenZipFile(train_dataset_archive, mode='r') as archive:
		archive.extractall(path=train_dataset_path)

In [3]:
####################################################################
# Set up the image cleaning pipeline
####################################################################
prep_pipeline = transforms.Compose([
	
	# Convert raw image to a tensor
	transforms.ToImage(),
	
	# Handle noise by applying a gaussian blur
	# we're not going to do that on a 32x32 image though
	# transforms.GaussianBlur(kernel_size=(3, 3), sigma=1.0),

	# Handle blur by randomly adjusting the sharpness  to prevent overfitting
	# we're not gonna actually do this either though
	# transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.5)

	# Hanld occlusions by adding them to the training images so the model is used to occluded data
	# again, not doing this on a 32x32 image
	# transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), value='random')
	
	# Normalioze image sizes
	transforms.Resize((32, 32)),

	# Add a neutral color padding around the image and then randomly crop it back down to 
	# 32x32 so the model can train on off-center data
	transforms.Pad(padding=4, fill=0),
	transforms.RandomCrop((32, 32)),

	# scale pixel values from [0, 255] down to [0.0, 1.0]
	transforms.ToDtype(torch.float32, scale=True)
])

In [4]:
####################################################################
# Iterate over ever file in the training dataset
####################################################################
processed_tensors_list = []
train_dataset_inner_path = os.path.join(train_dataset_path, 'train')
for file_name in os.listdir(train_dataset_inner_path):
	img_filepath = os.path.join(train_dataset_inner_path, file_name)
	if os.path.isfile(img_filepath):
		
		try:
			# Handle "null or erroneous" images
			raw_image = Image.open(img_filepath)
			raw_image.verify()
			
			# verify() will clsoe the iamge so we need to reopen it
			raw_image = Image.open(img_filepath) 
			final_tensor = prep_pipeline(raw_image)
			processed_tensors_list.append(final_tensor)
			
		except (IOError, SyntaxError) as e:
			print(f"skipping null/erroneous image: '{file_name}'")
			continue

# save the training tensor
master_image_tensor = torch.stack(processed_tensors_list, dim=0)
torch.save(master_image_tensor, 'x_train.pt')
